# GEO Testing Platform - Initial Testing

**Purpose**: Test Ollama connection, run single GEO transformations, and calculate metrics

**Date**: October 8, 2025

**Domain**: geooptimizer.dev

## 1. Setup and Imports

In [ ]:
import sys
import os
import asyncio
import json
from datetime import datetime

# Add backend to path
sys.path.insert(0, '../backend')

from services.ollama_client import OllamaClient
from services.geo_methods import GEOMethod, get_transformation_prompt, get_method_description
from services.metrics_calculator import GEOMetricsCalculator

print("✅ Imports successful")

## 2. Initialize Clients

In [ ]:
# Initialize Ollama client
ollama = OllamaClient(
    base_url="http://localhost:11434",
    default_model="mistral:7b-instruct"
)

# Initialize metrics calculator
calc = GEOMetricsCalculator()

print("✅ Clients initialized")

## 3. Test Ollama Connection

In [ ]:
# Check if Ollama is running
is_healthy = await ollama.check_health()
print(f"Ollama Health: {'✅ Running' if is_healthy else '❌ Not running'}")

if is_healthy:
    # List available models
    models = await ollama.list_models()
    print(f"\nAvailable models:")
    for model in models:
        print(f"  • {model}")

## 4. Define Test Query and Content

In [ ]:
# Test query
test_query = "What are the best Italian restaurants in New York City?"

# Baseline content (unoptimized)
baseline_content = """
Joe's Pizza has been serving authentic Italian pizza in New York City since 1975. 
Located in Greenwich Village, the restaurant offers traditional Neapolitan-style pizza with fresh ingredients.
The menu includes classic margherita, pepperoni, and specialty pizzas.
Customer reviews consistently praise the quality of ingredients and authentic taste.
The restaurant maintains a casual, family-friendly atmosphere.
""".strip()

# Competing sources
target_source = "Joe's Pizza"
competing_sources = [
    "Joe's Pizza",
    "Carbone Restaurant", 
    "L'Artusi",
    "Torrisi Bar & Restaurant"
]

print(f"Query: {test_query}")
print(f"\nTarget Source: {target_source}")
print(f"\nBaseline Content ({len(baseline_content.split())} words):")
print(baseline_content)

## 5. Test Single GEO Method: Statistics Addition

In [ ]:
# Select method
method = GEOMethod.STATISTICS

print(f"Testing: {get_method_description(method)}\n")

# Get transformation prompt
prompt_data = get_transformation_prompt(method, baseline_content)

print("System Prompt:")
print(prompt_data['system'])
print("\nUser Prompt:")
print(prompt_data['prompt'][:300] + "...")

In [ ]:
# Generate optimized content
print("Generating optimized content...")

result = await ollama.generate(
    prompt=prompt_data['prompt'],
    system=prompt_data['system'],
    temperature=0.7,
    model="mistral:7b-instruct"
)

optimized_content = result['text']
generation_time = result['generation_time']

print(f"\n✅ Generated in {generation_time}ms\n")
print("Optimized Content:")
print(optimized_content)

## 6. Generate AI Response (Simulated Generative Engine)

In [ ]:
# Create simulated generative engine prompt
# This simulates ChatGPT/Perplexity/Claude responding to a query with sources

ge_system_prompt = """
You are a helpful AI assistant answering user queries.
You have access to multiple sources of information.
Cite your sources when providing information.
Use the format: "According to [Source Name], ..." or "[Source Name] states that..."
"""

ge_user_prompt = f"""
Query: {test_query}

Available Sources:

Source 1: {target_source}
{optimized_content}

Source 2: Carbone Restaurant
Carbone is an upscale Italian-American restaurant in Greenwich Village serving classic red-sauce dishes in a 1950s-style setting.

Source 3: L'Artusi
L'Artusi offers modern Italian cuisine in the West Village with seasonal ingredients and an extensive wine list.

Source 4: Torrisi Bar & Restaurant  
Torrisi serves innovative Italian-American cuisine in a sleek, modern setting in Soho.

Please answer the query using these sources. Cite your sources when making claims.
"""

print("Generating AI response...")

response = await ollama.generate(
    prompt=ge_user_prompt,
    system=ge_system_prompt,
    temperature=0.7,
    model="mistral:7b-instruct"
)

ai_response = response['text']

print(f"\n✅ AI Response Generated\n")
print(ai_response)

## 7. Calculate GEO Metrics

In [ ]:
# Calculate metrics
metrics = calc.calculate_all_metrics(
    response_text=ai_response,
    target_source=target_source,
    source_titles=competing_sources
)

# Print formatted report
print(calc.format_metrics_report(metrics))

# Save metrics
metrics_file = f"../results/metrics_{method.value}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
with open(metrics_file, 'w') as f:
    json.dump(metrics, f, indent=2, default=str)
    
print(f"\n💾 Metrics saved to: {metrics_file}")

## 8. Compare Against Baseline

In [ ]:
# Run baseline (unoptimized) for comparison
baseline_prompt_data = get_transformation_prompt(GEOMethod.BASELINE, baseline_content)

baseline_optimized = await ollama.generate(
    prompt=baseline_prompt_data['prompt'],
    system=baseline_prompt_data['system'],
    temperature=0.7
)

# Generate AI response for baseline
baseline_ge_prompt = ge_user_prompt.replace(optimized_content, baseline_optimized['text'])
baseline_response = await ollama.generate(
    prompt=baseline_ge_prompt,
    system=ge_system_prompt,
    temperature=0.7
)

# Calculate baseline metrics
baseline_metrics = calc.calculate_all_metrics(
    response_text=baseline_response['text'],
    target_source=target_source,
    source_titles=competing_sources
)

print("BASELINE METRICS:")
print(calc.format_metrics_report(baseline_metrics))

In [ ]:
# Calculate improvement
optimized_metrics = calc.calculate_all_metrics(
    response_text=ai_response,
    target_source=target_source,
    source_titles=competing_sources,
    baseline_metrics=baseline_metrics
)

print("\n" + "="*60)
print(f"IMPROVEMENT ANALYSIS: {method.value.upper()} vs BASELINE")
print("="*60)

improvements = optimized_metrics.get('improvements', {})
for metric, improvement in improvements.items():
    symbol = "🔼" if improvement > 0 else "🔽"
    print(f"{symbol} {metric.upper()}: {improvement:+.1f}%")

# Save comparison
comparison = {
    "method": method.value,
    "baseline": baseline_metrics,
    "optimized": optimized_metrics,
    "improvements": improvements,
    "timestamp": datetime.now().isoformat()
}

comparison_file = f"../results/comparison_{method.value}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
with open(comparison_file, 'w') as f:
    json.dump(comparison, f, indent=2, default=str)
    
print(f"\n💾 Comparison saved to: {comparison_file}")

## 9. Test All Methods (Quick)

In [ ]:
# Test all GEO methods and compare
all_results = {}

print("Testing all GEO methods...\n")

for method in GEOMethod:
    print(f"Testing {method.value}...")
    
    # Get prompt
    prompt_data = get_transformation_prompt(method, baseline_content)
    
    # Generate optimized content
    result = await ollama.generate(
        prompt=prompt_data['prompt'],
        system=prompt_data['system'],
        temperature=0.7
    )
    
    # Generate AI response
    ge_prompt = ge_user_prompt.replace(optimized_content, result['text'])
    response = await ollama.generate(
        prompt=ge_prompt,
        system=ge_system_prompt,
        temperature=0.7
    )
    
    # Calculate metrics
    metrics = calc.calculate_all_metrics(
        response_text=response['text'],
        target_source=target_source,
        source_titles=competing_sources,
        baseline_metrics=baseline_metrics
    )
    
    all_results[method.value] = metrics
    
print("\n✅ All methods tested")

In [ ]:
# Compare all methods
import pandas as pd

comparison_data = []
for method, metrics in all_results.items():
    improvements = metrics.get('improvements', {})
    comparison_data.append({
        'Method': method,
        'PAWC': metrics['pawc'],
        'PAWC Improvement': improvements.get('pawc', 0),
        'Cited': '✓' if metrics['cited'] else '✗',
        'Position': metrics['citation_position'] or 'N/A'
    })

df = pd.DataFrame(comparison_data)
df = df.sort_values('PAWC Improvement', ascending=False)

print("\n" + "="*70)
print("METHOD COMPARISON - RANKED BY PAWC IMPROVEMENT")
print("="*70)
print(df.to_string(index=False))

# Save results
df.to_csv(f"../results/method_comparison_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv", index=False)
print(f"\n💾 Results saved")

## 10. Next Steps

✅ Ollama connection working  
✅ GEO methods implemented  
✅ Metrics calculation verified  
✅ Single query tested  

**Next Notebook**: `02_full_experiment.ipynb`  
- Test 50 queries across all domains
- Run 10 iterations per method
- Statistical significance testing
- Generate patent documentation